# 07 – Save the chosen model in MLflow (M3)

The M3 rubric asks for **the chosen model saved in MLflow and loadable from its run**,
with the run id in the README. The R7 runs logged scores only, so this notebook saves the
model itself.

- **Model:** M2 (chosen in `05_validation`), trained on all training students.
- **Saved as a pipeline:** fill missing prior GPA with the training median, then GaussianNB.
  This gives exactly the same predictions as `05_validation`.
- **The test set is not scored again.** We log the R8 test result from
  `results/test_result.csv` so the run shows it.

**Run this on the lab machine** (the course MLflow server is only reachable there).

## Step 1 – Setup

In [1]:
import sys
sys.path.append("..")

import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from src.split import load_features, get_train_test

# Course server (same as 04_models). Can be changed with the MLFLOW_TRACKING_URI variable.
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://10.0.100.20:5000/"))
mlflow.set_experiment("tutoring-paradox-r7")

df = load_features()
train, test = get_train_test(df)
print("Train rows:", len(train))

Train rows: 95991


## Step 2 – Train M2 as one pipeline

In [2]:
FEATURES = ["failed_t1", "bus_rider", "econ_disadvantaged", "english_learner", "iep",
            "subj_English Language Arts", "subj_Mathematics", "subj_Science",
            "prior_year_gpa", "gpa_missing"]

def prepare(data):
    """Same preparation as R7/R8: subject -> 0/1 columns, True/False -> 1/0."""
    X = pd.get_dummies(data, columns=["subject_area"], prefix="subj", dtype=int)
    for col in X.select_dtypes("bool").columns:
        X[col] = X[col].astype(int)
    return X

train_X = prepare(train)

# Step 1 of the pipeline fills missing GPA with the TRAINING median (same as 05_validation)
model = Pipeline([("fill_gpa", SimpleImputer(strategy="median")),
                  ("nb", GaussianNB())])
model.fit(train_X[FEATURES], train_X["failed_t3"])
print("Median GPA used for missing values:", model.named_steps["fill_gpa"].statistics_[8])

Median GPA used for missing values: 2.83


## Step 3 – Save it in MLflow

In [3]:
r8 = pd.read_csv("../results/test_result.csv").iloc[0]     # R8 test result (not re-scored)
example = train_X[FEATURES].head(5)

with mlflow.start_run(run_name="M2_final_model") as run:
    mlflow.log_params({"model_type": "GaussianNB pipeline (median GPA fill)",
                       "features": ",".join(FEATURES),
                       "chosen_in": "05_validation (highest CV AUC, lead 6x SD)",
                       "split_seed": 42})
    mlflow.log_metrics({"cv_auc": r8["cv_auc"], "test_auc": r8["test_auc"],
                        "test_brier": r8["test_brier"], "top20_catches": r8["top20_catches"]})
    mlflow.sklearn.log_model(model, "model", serialization_format="cloudpickle",
                             signature=infer_signature(example, model.predict_proba(example)),
                             input_example=example)
    run_id = run.info.run_id

print("Run id:", run_id)

/home/csc-4933-002-user06/venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/16 01:19:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/16 01:19:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exerc

🏃 View run M2_final_model at: http://10.0.100.20:5000/#/experiments/130/runs/6f90acd9ca49441aab6266696cdd3a19
🧪 View experiment at: http://10.0.100.20:5000/#/experiments/130
Run id: 6f90acd9ca49441aab6266696cdd3a19


## Step 4 – Load it back and check it works
We load the model from its run and check that it gives exactly the same predictions as
the model we just trained.

In [4]:
loaded = mlflow.sklearn.load_model(f"runs:/{run_id}/model")

check = train_X[FEATURES].head(1000)
same = np.allclose(loaded.predict_proba(check), model.predict_proba(check))
print("Loaded from:", f"runs:/{run_id}/model")
print("Same predictions as the trained model:", same)

# Save the run id so the README can point to it
Path = __import__("pathlib").Path
Path("../results").mkdir(exist_ok=True)
Path("../results/model_run_id.txt").write_text(run_id + "\n")
print("Saved ../results/model_run_id.txt")

Loaded from: runs:/6f90acd9ca49441aab6266696cdd3a19/model
Same predictions as the trained model: True
Saved ../results/model_run_id.txt


**Result:** M2 is saved in MLflow as run `M2_final_model`. Anyone on the lab network can
load it with:

```python
model = mlflow.sklearn.load_model("runs:/<run id>/model")
model.predict_proba(X)   # X = the 10 feature columns, prepared as in this notebook
```

The run id is in `results/model_run_id.txt` and in the README.